# Otso Vision v2 - Binarization Model Training

Notebook ini dikonfigurasi untuk Google Colab GPU. Jalankan dari atas ke bawah setelah memilih Runtime type: GPU.

Target output: `otso_docclean_v1.tflite` dengan kontrak Android:
- input: `[1, 256, 256, 3]` `float32`, RGB normalized `0..1`
- output: `[1, 256, 256, 1]` `float32`, background probability mask


## 1. Setup Environment

In [ ]:
!pip install -q tensorflow pillow matplotlib

import os
import glob
import shutil
import zipfile
from pathlib import Path

import numpy as np
import tensorflow as tf
from PIL import Image, ImageFile, UnidentifiedImageError

ImageFile.LOAD_TRUNCATED_IMAGES = True

tf.keras.utils.set_random_seed(42)
print('TensorFlow:', tf.__version__)


## 2. Smart Dataset Sync

Download dataset dari endpoint Home Server. Jika file zip sudah ada di Colab, `wget -c` akan melanjutkan/resume.

In [ ]:
DATASET_URL = 'https://unittesting01.krtalabs.xyz/otso_datasets_v2.zip'
ZIP_FILE = Path('/content/otso_datasets.zip')
EXTRACT_DIR = Path('/content/otso_datasets')

print('Checking dataset archive...')
!wget -c {DATASET_URL} -O {ZIP_FILE}

if not ZIP_FILE.exists() or ZIP_FILE.stat().st_size == 0:
    raise RuntimeError(f'Dataset zip missing or empty: {ZIP_FILE}')

print('Testing zip integrity...')
with zipfile.ZipFile(ZIP_FILE, 'r') as zf:
    bad_member = zf.testzip()
    if bad_member is not None:
        raise RuntimeError(f'Corrupt zip member: {bad_member}')

print('Extracting dataset...')
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_FILE, 'r') as zf:
    zf.extractall(EXTRACT_DIR)

print('Sync complete:', EXTRACT_DIR)


## 3. Data Pipeline With Image Sanitization

Cell ini memindai dataset, memverifikasi gambar dengan PIL, lalu membuat salinan PNG 256x256 yang bersih. Training memakai folder sanitized ini agar `tf.io.decode_png()` tidak crash di tengah epoch.

In [ ]:
IMG_SIZE = (256, 256)
BATCH_SIZE = 16
MAX_IMAGES = 12000
MAX_BAD_IMAGE_LOG = 25
SANITIZED_DIR = Path('/content/otso_clean_images')
SUPPORTED_EXTS = {'.png', '.jpg', '.jpeg', '.webp', '.bmp', '.tif', '.tiff'}

print('Scanning extracted dataset...')
search_roots = [EXTRACT_DIR, Path('/content')]
candidate_images = []
for root in search_roots:
    if root.exists():
        for path in root.rglob('*'):
            if path.is_file() and path.suffix.lower() in SUPPORTED_EXTS:
                candidate_images.append(path)

candidate_images = sorted(set(candidate_images))
print('Candidate image files:', len(candidate_images))
if not candidate_images:
    print('Top-level /content:', os.listdir('/content'))
    raise RuntimeError('No candidate image files found after extraction.')

if SANITIZED_DIR.exists():
    shutil.rmtree(SANITIZED_DIR)
SANITIZED_DIR.mkdir(parents=True, exist_ok=True)

valid_images = []
bad_images = []
for index, path in enumerate(candidate_images[:MAX_IMAGES]):
    try:
        if path.stat().st_size <= 0:
            raise ValueError('zero-byte file')
        with Image.open(path) as img:
            img = img.convert('RGB')
            img = img.resize(IMG_SIZE, Image.Resampling.BILINEAR)
            target = SANITIZED_DIR / f'{len(valid_images):06d}.png'
            img.save(target, format='PNG', optimize=False)
            valid_images.append(str(target))
    except Exception as error:
        bad_images.append((str(path), str(error)))

print('Valid sanitized images:', len(valid_images))
print('Bad images skipped:', len(bad_images))
for path, error in bad_images[:MAX_BAD_IMAGE_LOG]:
    print(f'SKIP_BAD_IMAGE: {path} :: {error}')
if len(bad_images) > MAX_BAD_IMAGE_LOG:
    print(f'... {len(bad_images) - MAX_BAD_IMAGE_LOG} more bad images omitted from log')

if len(valid_images) < 200:
    raise RuntimeError(f'Too few valid images for training: {len(valid_images)}')

def load_and_preprocess(img_path):
    img_bytes = tf.io.read_file(img_path)
    img = tf.io.decode_png(img_bytes, channels=3)
    img = tf.cast(img, tf.float32) / 255.0
    gray = tf.image.rgb_to_grayscale(img)
    # Pseudo-label: white/background=1.0, dark ink=0.0.
    mask = tf.where(gray < 0.5, 0.0, 1.0)
    return img, mask

train_ds = tf.data.Dataset.from_tensor_slices(valid_images)
train_ds = train_ds.shuffle(len(valid_images), reshuffle_each_iteration=True)
train_ds = train_ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

sample_batch = next(iter(train_ds.take(1)))
print('Sample input batch:', sample_batch[0].shape, sample_batch[0].dtype)
print('Sample mask batch:', sample_batch[1].shape, sample_batch[1].dtype)
print('Neural Pipeline READY.')


## 4. MobileNetV3-Small Binarization Model

In [ ]:
def build_model():
    # Android NeuralVisionEngine sends float32 RGB normalized to 0..1.
    # MobileNetV3 preprocessing expects 0..255, so scale inside the model.
    inputs = tf.keras.Input(shape=(256, 256, 3), name='rgb_normalized')
    x = tf.keras.layers.Lambda(lambda t: t * 255.0, name='to_mobilenet_range')(inputs)
    base_model = tf.keras.applications.MobileNetV3Small(
        input_shape=(256, 256, 3),
        include_top=False,
        weights='imagenet',
        include_preprocessing=True,
    )
    base_model.trainable = False

    x = base_model(x)
    x = tf.keras.layers.Conv2DTranspose(64, 3, strides=2, padding='same', activation='relu')(x)
    x = tf.keras.layers.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(x)
    x = tf.keras.layers.Conv2DTranspose(16, 3, strides=2, padding='same', activation='relu')(x)
    x = tf.keras.layers.Conv2DTranspose(8, 3, strides=2, padding='same', activation='relu')(x)
    x = tf.keras.layers.Conv2DTranspose(4, 3, strides=2, padding='same', activation='relu')(x)
    outputs = tf.keras.layers.Conv2D(1, 3, padding='same', activation='sigmoid', name='background_probability')(x)
    model = tf.keras.Model(inputs, outputs, name='otso_docclean_v1')
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    return model

model = build_model()
model.summary()
history = model.fit(train_ds, epochs=5)


## 5. TFLite Conversion - Safe Float32 Export

Default export sengaja full-float32. Jangan aktifkan `tf.lite.Optimize.DEFAULT` di cell ini, karena export sebelumnya gagal di Android dengan mismatch `INT8 != FLOAT32` pada `TRANSPOSE_CONV`.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = []
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
tflite_model = converter.convert()

OUTPUT_NAME = 'otso_docclean_v1.tflite'
with open(OUTPUT_NAME, 'wb') as f:
    f.write(tflite_model)

interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

print('Wrote', OUTPUT_NAME, 'bytes=', len(tflite_model))
print('input', input_details['shape'], input_details['dtype'])
print('output', output_details['shape'], output_details['dtype'])

assert tuple(input_details['shape']) == (1, 256, 256, 3)
assert tuple(output_details['shape']) == (1, 256, 256, 1)
assert input_details['dtype'] == np.float32
assert output_details['dtype'] == np.float32
assert len(tflite_model) <= 5 * 1024 * 1024, 'Model exceeds 5 MB PRD limit.'

# Smoke inference inside Colab.
test_input = np.zeros((1, 256, 256, 3), dtype=np.float32)
interpreter.set_tensor(input_details['index'], test_input)
interpreter.invoke()
test_output = interpreter.get_tensor(output_details['index'])
print('smoke output:', test_output.shape, test_output.dtype, float(test_output.min()), float(test_output.max()))

from google.colab import files
files.download(OUTPUT_NAME)


## 6. Download Existing Export

In [ ]:
from google.colab import files
OUTPUT_NAME = 'otso_docclean_v1.tflite'
if os.path.exists(OUTPUT_NAME):
    files.download(OUTPUT_NAME)
else:
    print(f'File tidak ditemukan di Colab: {OUTPUT_NAME}. Jalankan cell export terlebih dahulu.')
